<a href="https://colab.research.google.com/github/Maximi652/efficient-slm-architectures/blob/main/WANDA_Qweb3_4B_C.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# WANDA Pruning Implementation for Qwen3-4B
# Applies Weight-and-Activation-aware pruning to reduce model size
# Based on unstructured pruning per output/row

# Installation and Setup
!pip -q install "transformers>=4.51.0" "accelerate>=0.33.0" "torch>=2.1" pandas sentencepiece

import os
import json
import math
import random
import gc
from dataclasses import dataclass
from typing import List, Dict, Tuple, Set
from collections import Counter
import re

import torch
import torch.nn as nn
import pandas as pd
from transformers import AutoTokenizer, AutoModelForCausalLM

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# Mount Google Drive and set paths
from google.colab import drive
drive.mount('/content/drive')

TRAIN_PATH = "/content/drive/MyDrive/Colab Notebooks/12B_trainingdata.json"
TEST_PATH = "/content/drive/MyDrive/Colab Notebooks/12B_golden_testdata.json"

# Use test data as fallback if training data not available
if not os.path.exists(TRAIN_PATH):
    if os.path.exists(TEST_PATH):
        TRAIN_PATH = TEST_PATH
        print("Training data not found - using test data for calibration")
    else:
        raise FileNotFoundError("Neither training nor test data found")

# Data Loading and Processing
def load_bioasq_questions(path: str) -> List[Dict]:
    """Load BioASQ questions from JSON file"""
    with open(path, "r") as f:
        data = json.load(f)
    return data["questions"]

def build_prompt(q):
    """Build compact prompt for calibration"""
    t = q.get("type", "")
    body = q.get("body", "").strip()
    head = "Beantworte medizinische Frage knapp und sachlich."

    if t == "yesno":
        instr = "Antworte mit Ja oder Nein und maximal einem kurzen Satz."
    elif t == "list":
        instr = "Gib eine knappe Liste der wichtigsten Punkte aus."
    elif t == "factoid":
        instr = "Gib die gesuchte Entität in 1-2 Wörtern an."
    else:
        instr = "Antworte kurz."

    return f"{head}\nFrage: {body}\n{instr}"

def encode_batch(texts: List[str], tokenizer, max_len=256):
    """Encode text batch using Qwen3 chat template"""
    rendered = [
        tokenizer.apply_chat_template(
            [{"role":"user","content":t}],
            tokenize=False,
            add_generation_prompt=False,
            enable_thinking=False
        )
        for t in texts
    ]
    toks = tokenizer(
        rendered, return_tensors="pt",
        padding=True, truncation=True, max_length=max_len
    )
    return toks

# Load and prepare calibration data
all_questions = load_bioasq_questions(TRAIN_PATH)
random.seed(42)
random.shuffle(all_questions)

CALIB_SAMPLES = 64      # Reduce to 32/16 if OOM
MAX_PROMPT_TOKENS = 256
calib_texts = [build_prompt(q) for q in all_questions[:CALIB_SAMPLES]]

# Load Qwen3-4B model
MODEL_ID = "Qwen/Qwen3-4B"
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

# Load model with memory optimization
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map="auto",
    low_cpu_mem_usage=True
)
model.eval()
if hasattr(model.config, "use_cache"):
    model.config.use_cache = False  # Save RAM during forward pass

# Calculate original model metrics
def calculate_model_metrics(model):
    """Calculate model size and parameter statistics"""
    total_params = 0
    zero_params = 0
    model_size_bytes = 0

    for param in model.parameters():
        total_params += param.numel()
        zero_params += (param == 0).sum().item()
        model_size_bytes += param.numel() * param.element_size()

    sparsity = zero_params / total_params if total_params > 0 else 0
    model_size_gb = model_size_bytes / (1024**3)

    return {
        "total_params": total_params,
        "zero_params": zero_params,
        "sparsity": sparsity,
        "model_size_gb": model_size_gb
    }

original_metrics = calculate_model_metrics(model)
print(f"Original Model - Size: {original_metrics['model_size_gb']:.2f} GB, "
      f"Total params: {original_metrics['total_params']:,}, "
      f"Sparsity: {original_metrics['sparsity']:.2%}")

# WANDA Calibration Setup
@dataclass
class LayerStats:
    """Statistics for layer activation norms"""
    sumsq: torch.Tensor  # Sum of squares per input feature (on CPU)
    in_features: int

def iter_linear_modules(model: nn.Module):
    """Iterate over linear modules excluding embedding and language model head"""
    skip_names = ("lm_head", "embed_tokens")
    for name, module in model.named_modules():
        if isinstance(module, nn.Linear) and not any(s in name for s in skip_names):
            yield name, module

# Prepare statistics storage
layer_stats: Dict[str, LayerStats] = {}
for name, mod in iter_linear_modules(model):
    layer_stats[name] = LayerStats(
        sumsq=torch.zeros(mod.in_features, dtype=torch.float64, device="cpu"),
        in_features=mod.in_features
    )

# Hook function to accumulate activation statistics
def make_hook(name: str):
    """Create hook to accumulate sum of squares over batch*sequence"""
    def hook(module, inputs, output):
        x = inputs[0]
        if x.dim() == 3:
            x = x.reshape(-1, x.size(-1))
        # Calculate on GPU, then add to CPU (only vector size C_in)
        ss = (x.float() ** 2).sum(dim=0).detach().cpu()
        layer_stats[name].sumsq += ss
        return None
    return hook

# Register hooks
handles = []
for name, mod in iter_linear_modules(model):
    handles.append(mod.register_forward_hook(make_hook(name)))

# Run calibration - small batches, no KV cache, no gradients
BATCH_SIZE = 2  # Reduce to 1 if OOM
print("Running WANDA calibration...")
with torch.no_grad():
    for i in range(0, len(calib_texts), BATCH_SIZE):
        batch = calib_texts[i:i+BATCH_SIZE]
        toks = encode_batch(batch, tokenizer)
        toks = {k: v.to(model.device) for k, v in toks.items()}
        _ = model(**toks)  # Forward pass only
        del toks
        torch.cuda.empty_cache()

# Remove hooks and clean up
for h in handles:
    h.remove()
del handles
gc.collect()
torch.cuda.empty_cache()

# Apply WANDA Pruning
SPARSITY = 0.30  # 30% of weights per row set to 0
EPS = 1e-12

print("Applying WANDA pruning...")
total_params = 0
total_zero = 0

for name, mod in iter_linear_modules(model):
    W = mod.weight.data  # Shape: (C_out, C_in)
    Cout, Cin = W.shape
    stats = layer_stats[name]

    # L2 norm over tokens (sqrt of sum of squares)
    act_norm = torch.sqrt(stats.sumsq + EPS)  # Shape: (C_in,)
    act_norm = act_norm.to(W.device, dtype=torch.float32)

    # WANDA metric: |W| * ||X||_2, compare per output
    metric = W.abs().to(torch.float32) * act_norm.unsqueeze(0)  # Shape: (C_out, C_in)

    k_prune = int(Cin * SPARSITY)
    if k_prune == 0:
        continue

    # Find smallest scores per row
    prune_idx = torch.argsort(metric, dim=1)[:, :k_prune]  # Shape: (C_out, k)

    # Build mask: True=keep, False=prune
    mask = torch.ones_like(W, dtype=torch.bool, device=W.device)
    mask.scatter_(1, prune_idx, False)

    # Apply pruning
    W[~mask] = 0

    total_params += W.numel()
    total_zero += (~mask).sum().item()

sparsity_achieved = total_zero / max(1, total_params)
print(f"Pruning completed. Global unstructured sparsity: {sparsity_achieved:.2%}")

# Calculate post-pruning metrics
pruned_metrics = calculate_model_metrics(model)
compression_ratio = original_metrics['model_size_gb'] / pruned_metrics['model_size_gb']

# Save pruned model
OUT_DIR = f"/content/drive/MyDrive/Colab Notebooks/qwen3-4b-wanda-{int(SPARSITY*100)}p"
os.makedirs(OUT_DIR, exist_ok=True)
model.save_pretrained(OUT_DIR, safe_serialization=True)
tokenizer.save_pretrained(OUT_DIR)
print(f"Model saved to: {OUT_DIR}")

# Print comprehensive metrics
print("\n" + "="*60)
print("MODEL METRICS COMPARISON")
print("="*60)
print(f"{'Metric':<25} {'Original':<15} {'Pruned':<15} {'Change':<15}")
print("-"*60)
print(f"{'Size (GB)':<25} {original_metrics['model_size_gb']:<15.3f} "
      f"{pruned_metrics['model_size_gb']:<15.3f} "
      f"{((pruned_metrics['model_size_gb']/original_metrics['model_size_gb']-1)*100):+.1f}%")
print(f"{'Total Parameters':<25} {original_metrics['total_params']:<15,} "
      f"{pruned_metrics['total_params']:<15,} {'0':<15}")
print(f"{'Zero Parameters':<25} {original_metrics['zero_params']:<15,} "
      f"{pruned_metrics['zero_params']:<15,} "
      f"{pruned_metrics['zero_params']-original_metrics['zero_params']:+,}")
print(f"{'Sparsity':<25} {original_metrics['sparsity']:<15.2%} "
      f"{pruned_metrics['sparsity']:<15.2%} "
      f"{(pruned_metrics['sparsity']-original_metrics['sparsity'])*100:+.1f}pp")
print(f"{'Compression Ratio':<25} {'1.00x':<15} {compression_ratio:<15.2f}x "
      f"{compression_ratio-1:+.2f}x")

# Sanity check with short generation
print("\n" + "="*60)
print("SANITY CHECK")
print("="*60)
example = (build_prompt(all_questions[CALIB_SAMPLES])
           if len(all_questions) > CALIB_SAMPLES
           else "Nenne die Symptome einer leichten Erkältung in 3 Stichpunkten.")

inputs = tokenizer.apply_chat_template(
    [{"role":"user","content":example}],
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=False
)
toks = tokenizer([inputs], return_tensors="pt").to(model.device)

with torch.no_grad():
    out = model.generate(**toks, max_new_tokens=64, do_sample=False)
result = tokenizer.decode(out[0], skip_special_tokens=True)
print("Sample generation:")
print(result)

# Evaluation Functions
def normalize(s: str) -> str:
    """Normalize text for matching"""
    s = s.strip().lower()
    s = re.sub(r"\s+", " ", s)
    s = re.sub(r"[^\w\s\-/%]", "", s)  # Remove punctuation
    s = s.strip(" .,:;!?\"'()[]{}")
    return s

def norm_set(items: List[str]) -> Set[str]:
    """Create normalized set of items"""
    return {normalize(x) for x in items if str(x).strip()}

def match_any(pred: str, gold_syns: List[str]) -> bool:
    """Check if prediction matches any gold synonym"""
    p = normalize(pred)
    gset = norm_set(gold_syns)
    return p in gset

def split_pred_list(text: str) -> List[str]:
    """Split prediction text into list items"""
    parts = re.split(r"[,\n;]+", text)
    parts = [p.strip() for p in parts if p.strip()]
    # Remove duplicates and very short artifacts
    uniq = []
    seen = set()
    for p in parts:
        n = normalize(p)
        if len(n) == 0 or n in seen:
            continue
        seen.add(n)
        uniq.append(p)
    return uniq

def render_user(msg: str, tokenizer) -> str:
    """Render user message with chat template"""
    return tokenizer.apply_chat_template(
        [{"role":"user","content":msg}],
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False
    )

def generate_once(prompt: str, model, tokenizer, max_new_tokens=8,
                 do_sample=False, temperature=0.7, top_p=0.9) -> str:
    """Generate single response"""
    text = render_user(prompt, tokenizer)
    toks = tokenizer([text], return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(
            **toks,
            max_new_tokens=max_new_tokens,
            do_sample=do_sample,
            temperature=temperature,
            top_p=top_p,
            pad_token_id=tokenizer.eos_token_id
        )
    ans = tokenizer.decode(out[0], skip_special_tokens=True)
    return ans.split(text)[-1].strip()  # Assistant response only

def yn_label_from_text(s: str) -> str:
    """Extract yes/no label from text"""
    s = normalize(s)
    if s.startswith("yes") or s in {"y", "yeah", "yep", "true"}:
        return "yes"
    if s.startswith("no") or s in {"n", "nope", "false"}:
        return "no"
    # Fallback heuristic
    return "yes" if "yes" in s else "no" if "no" in s else "no"

def f1_score(p, r):
    """Calculate F1 score"""
    return 0.0 if (p+r)==0 else 2*p*r/(p+r)

def eval_yesno(preds: List[str], golds: List[str]) -> Dict[str, float]:
    """Evaluate yes/no predictions"""
    labels = ["yes","no"]
    cm = {c: {"tp":0, "fp":0, "fn":0, "tn":0} for c in labels}
    correct = 0

    for pr, gd in zip(preds, golds):
        if pr == gd:
            correct += 1
        for c in labels:
            tp = (pr==c and gd==c)
            fp = (pr==c and gd!=c)
            fn = (pr!=c and gd==c)
            tn = (pr!=c and gd!=c)
            cm[c]["tp"] += tp
            cm[c]["fp"] += fp
            cm[c]["fn"] += fn
            cm[c]["tn"] += tn

    acc = correct/len(golds) if golds else 0.0
    out = {"Accuracy": acc}
    f1s = []

    for c in labels:
        P = cm[c]["tp"] / max(1, (cm[c]["tp"]+cm[c]["fp"]))
        R = cm[c]["tp"] / max(1, (cm[c]["tp"]+cm[c]["fn"]))
        out[f"F1 {c.capitalize()}"] = f1_score(P,R)
        f1s.append(out[f"F1 {c.capitalize()}"])

    out["Macro F1"] = sum(f1s)/len(f1s)
    return out

def eval_factoid(pred_topk: List[List[str]], gold_groups: List[List[str]]) -> Dict[str,float]:
    """Evaluate factoid predictions"""
    strict_hits = 0
    lenient_hits = 0
    rr_sum = 0.0
    n = len(pred_topk)

    for cand_list, gold in zip(pred_topk, gold_groups):
        gold_syns = [g for g in gold] if isinstance(gold[0], str) else [x for group in gold for x in group]
        rank = None

        for i, cand in enumerate(cand_list, start=1):
            if match_any(cand, gold_syns):
                rank = i
                break

        if rank == 1:
            strict_hits += 1
        if rank is not None:
            lenient_hits += 1
            rr_sum += 1.0/rank

    return {
        "Strict Acc.": strict_hits/max(1,n),
        "Lenient Acc.": lenient_hits/max(1,n),
        "MRR": rr_sum/max(1,n),
    }

def eval_list(pred_lists: List[List[str]], gold_groups_list: List[List[List[str]]]) -> Dict[str,float]:
    """Evaluate list predictions"""
    precs, recs, fms = [], [], []

    for preds, gold_groups in zip(pred_lists, gold_groups_list):
        gold_matched = [False]*len(gold_groups)
        correct = 0

        for p in preds:
            hit = False
            for gi, group in enumerate(gold_groups):
                if gold_matched[gi]:
                    continue
                if match_any(p, group):
                    gold_matched[gi] = True
                    hit = True
                    break
            if hit:
                correct += 1

        P = correct / max(1, len(preds))
        R = correct / max(1, len(gold_groups))
        precs.append(P)
        recs.append(R)
        fms.append(f1_score(P,R))

    return {
        "Mean Prec.": sum(precs)/max(1,len(precs)),
        "Recall": sum(recs)/max(1,len(recs)),
        "F-Measure": sum(fms)/max(1,len(fms)),
    }

# Load test data and run evaluation
if os.path.exists(TEST_PATH):
    print("\n" + "="*60)
    print("EVALUATION ON TEST DATA")
    print("="*60)

    with open(TEST_PATH, "r") as f:
        test_data = json.load(f)
    questions = test_data["questions"]

    counts = Counter(q["type"] for q in questions)
    print("Question types:", dict(counts))

    # Evaluation parameters
    random.seed(42)
    YESNO_MAX_NEW = 2
    FACTOID_MAX_NEW = 8
    LIST_MAX_NEW = 24
    FACTOID_TOPK = 5

    yn_gold, yn_pred = [], []
    fact_gold, fact_pred_topk = [], []
    list_gold, list_pred = [], []

    print("Generating predictions...")
    for q in questions:
        qtype = q["type"]
        body = q["body"].strip()

        if qtype == "yesno":
            gold = q["exact_answer"].strip().lower()
            prompt = (
                "Answer the biomedical yes/no question with a single word ONLY: 'yes' or 'no'.\n"
                f"Question: {body}\n"
                "Answer:"
            )
            out = generate_once(prompt, model, tokenizer, max_new_tokens=YESNO_MAX_NEW, do_sample=False)
            pr = yn_label_from_text(out)
            yn_gold.append(gold)
            yn_pred.append(pr)

        elif qtype == "factoid":
            gold_groups = q["exact_answer"]
            prompt = (
                "Answer the biomedical factoid question with ONLY the short entity name (1-3 words). "
                "No extra text, no punctuation.\n"
                f"Question: {body}\n"
                "Short answer:"
            )
            # Generate top-k candidates
            topk = []
            out1 = generate_once(prompt, model, tokenizer, max_new_tokens=FACTOID_MAX_NEW, do_sample=False)
            topk.append(out1)

            # Additional candidates with sampling
            for k in range(FACTOID_TOPK-1):
                outk = generate_once(prompt, model, tokenizer, max_new_tokens=FACTOID_MAX_NEW,
                                   do_sample=True, temperature=0.8, top_p=0.9)
                if normalize(outk) not in {normalize(x) for x in topk}:
                    topk.append(outk)

            fact_gold.append(gold_groups)
            fact_pred_topk.append(topk[:FACTOID_TOPK])

        elif qtype == "list":
            gold_groups = q["exact_answer"]
            prompt = (
                "List the biomedical items asked for, as a comma-separated list. "
                "Return ONLY the list, no explanations.\n"
                f"Question: {body}\n"
                "List:"
            )
            out = generate_once(prompt, model, tokenizer, max_new_tokens=LIST_MAX_NEW, do_sample=False)
            preds = split_pred_list(out)
            list_gold.append(gold_groups)
            list_pred.append(preds)

        # Clear cache periodically
        torch.cuda.empty_cache()

    print(f"Evaluated: {len(yn_gold)} yes/no, {len(fact_gold)} factoid, {len(list_gold)} list questions")

    # Calculate metrics
    metrics = {}
    if yn_gold:
        metrics.update(eval_yesno(yn_pred, yn_gold))
    if fact_gold:
        metrics.update(eval_factoid(fact_pred_topk, fact_gold))
    if list_gold:
        metrics.update(eval_list(list_pred, list_gold))

    # Create comparison table
    baseline = {
        "System": "BioASQ_Baseline",
        "Accuracy": 0.4706,
        "F1 Yes": 0.4000,
        "F1 No": 0.5263,
        "Macro F1": 0.4632,
        "Strict Acc.": 0.1538,
        "Lenient Acc.": 0.2692,
        "MRR": 0.1955,
        "Mean Prec.": 0.2503,
        "Recall": 0.2390,
        "F-Measure": 0.2202,
    }

    pruned_row = {
        "System": "Qwen3-4B-WANDA",
        "Accuracy": round(metrics.get("Accuracy", 0.0), 4),
        "F1 Yes": round(metrics.get("F1 Yes", 0.0), 4),
        "F1 No": round(metrics.get("F1 No", 0.0), 4),
        "Macro F1": round(metrics.get("Macro F1", 0.0), 4),
        "Strict Acc.": round(metrics.get("Strict Acc.", 0.0), 4),
        "Lenient Acc.": round(metrics.get("Lenient Acc.", 0.0), 4),
        "MRR": round(metrics.get("MRR", 0.0), 4),
        "Mean Prec.": round(metrics.get("Mean Prec.", 0.0), 4),
        "Recall": round(metrics.get("Recall", 0.0), 4),
        "F-Measure": round(metrics.get("F-Measure", 0.0), 4),
    }

    cols = ["System","Accuracy","F1 Yes","F1 No","Macro F1","Strict Acc.",
            "Lenient Acc.","MRR","Mean Prec.","Recall","F-Measure"]
    df = pd.DataFrame([baseline, pruned_row], columns=cols)

    # Format for German decimal separator
    def format_de(x):
        if isinstance(x, float):
            return f"{x:.4f}".replace(".", ",")
        return x

    df_display = df.copy()
    for c in cols[1:]:
        df_display[c] = df_display[c].map(format_de)

    print("\nBioASQ Metrics (Baseline vs. Pruned):")
    print(df_display.to_string(index=False))
else:
    print("Test data not available for evaluation")

print("\n" + "="*60)
print("PRUNING COMPLETED SUCCESSFULLY")
print("="*60)